# EDA For Ecommerce Recommendations

Ноутбук фиксирует исследовательскую часть проекта как самостоятельный review-ready артефакт. Цель анализа: понять, какой recommendation setup даёт на этом датасете максимальную практическую отдачу при memory-safe ограничениях локального проекта.

В фокусе находится не весь массив просмотров, а вероятность будущего `addtocart`. Просмотры используются как контекст интереса, покупки — как более сильный позитивный сигнал и как источник исключений для уже купленных товаров.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

ROOT = Path.cwd().resolve()
if not (ROOT / 'artifacts').exists():
    ROOT = ROOT.parent

with (ROOT / 'eda_summary.json').open(encoding='utf-8') as f:
    eda_summary = json.load(f)
with (ROOT / 'artifacts' / 'reports' / 'eda_notebook_report.json').open(encoding='utf-8') as f:
    notebook_report = json.load(f)

events = pd.read_parquet(ROOT / 'artifacts' / 'data' / 'events_prepared.parquet', columns=['timestamp', 'visitorid', 'event', 'itemid'])
aggregated_train = pd.read_parquet(ROOT / 'artifacts' / 'data' / 'aggregated_train.parquet')
item_snapshot = pd.read_parquet(ROOT / 'artifacts' / 'data' / 'item_snapshot.parquet')

events['datetime'] = pd.to_datetime(events['timestamp'], unit='ms')
events['date'] = events['datetime'].dt.date
events['month'] = events['datetime'].dt.to_period('M').dt.to_timestamp()

train_end = pd.to_datetime(notebook_report['time_coverage']['train_end'])
validation_start = pd.to_datetime(notebook_report['time_coverage']['validation_start'])
events.shape, aggregated_train.shape, item_snapshot.shape


## 1. Данные и масштаб

Анализ строится на processed artifacts, а не на полном raw join. Это позволяет сохранить связь с реальным пайплайном: используются те же data products, которые дальше идут в offline evaluation, сервис и ETL.


In [ ]:
overview = pd.DataFrame(notebook_report['dataset_overview'])
overview['rows'] = overview['rows'].map(lambda x: f'{x:,}')

scale_table = pd.DataFrame([
    {'metric': 'unique_users_in_events', 'value': eda_summary['uniques']['users']},
    {'metric': 'unique_items_in_events', 'value': eda_summary['uniques']['items_in_events']},
    {'metric': 'unique_items_in_properties', 'value': eda_summary['uniques']['items_in_properties']},
    {'metric': 'unique_properties', 'value': eda_summary['uniques']['properties']},
    {'metric': 'transaction_ids', 'value': eda_summary['uniques']['transactions']},
])
scale_table['value'] = scale_table['value'].map(lambda x: f'{x:,}')

display(overview)
display(scale_table)


**Выводы блока**

- Проект работает с тремя источниками разного масштаба: компактный `events`, очень крупный `item_properties` и небольшое дерево категорий.
- Даже после фильтрации до нужных полей масштаб достаточен, чтобы требовать аккуратной работы с типами и промежуточными артефактами.
- Основной рабочий контур естественно строится вокруг событий, а свойства товаров выступают вторичным источником контекста.


## 2. Почему таргетируется именно `addtocart`

Для рекомендательной задачи нужен сигнал, который уже отражает намерение, но встречается чаще, чем покупка. На этом датасете `addtocart` как раз занимает промежуточную позицию между шумными просмотрами и редкими транзакциями.


In [ ]:
event_mix = (
    events['event']
    .value_counts(normalize=True)
    .rename_axis('event')
    .reset_index(name='share')
    .assign(share_pct=lambda df: df['share'] * 100)
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=event_mix, x='event', y='share_pct', hue='event', palette='Blues_r', legend=False, ax=ax)
ax.set_title('Структура событийного лога')
ax.set_xlabel('event')
ax.set_ylabel('share, %')
plt.tight_layout()

event_mix[['event', 'share_pct']]


**Выводы блока**

- `view` формирует почти весь поток событий и полезен как сигнал интереса, но не как основной target для evaluation.
- `addtocart` достаточно редок, чтобы быть содержательным бизнес-сигналом, и при этом существенно плотнее, чем `transaction`.
- `transaction` лучше трактовать как усиленный позитив и как источник фильтрации уже купленных товаров, а не как единственный таргет.


## 3. Временное покрытие и split discipline

Данные имеют выраженную временную структуру, поэтому исследование должно проверять не только объём, но и устойчивость покрытия по месяцам. Это напрямую определяет, можно ли использовать random split и насколько оправдан validation window на последних 14 днях.


In [ ]:
coverage = pd.DataFrame([
    {'range': 'events', 'start': notebook_report['time_coverage']['events_min'], 'end': notebook_report['time_coverage']['events_max']},
    {'range': 'item_properties', 'start': notebook_report['time_coverage']['properties_min'], 'end': notebook_report['time_coverage']['properties_max']},
    {'range': 'train window', 'start': notebook_report['time_coverage']['events_min'], 'end': notebook_report['time_coverage']['train_end']},
    {'range': 'validation window', 'start': notebook_report['time_coverage']['validation_start'], 'end': notebook_report['time_coverage']['events_max']},
])

monthly = (
    events.groupby(['month', 'event'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
monthly_long = monthly.melt(id_vars='month', var_name='event', value_name='events')

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=monthly_long, x='month', y='events', hue='event', marker='o', ax=ax)
ax.axvline(validation_start, color='black', linestyle='--', linewidth=1.5, label='validation start')
ax.set_title('Помесячная динамика событий и граница time-based split')
ax.set_xlabel('month')
ax.set_ylabel('events')
ax.legend()
plt.tight_layout()

coverage


**Выводы блока**

- Временной диапазон закрывает период с мая по середину сентября 2015 года, поэтому random split давал бы leakage между похожими по времени взаимодействиями.
- Validation действительно расположена на правом краю временной шкалы, а не внутри общего пула наблюдений.
- Monthly coverage выглядит достаточно ровным для time-based evaluation: резких провалов, которые ломали бы последний holdout, не видно.


## 4. Воронка `view -> addtocart -> transaction`

Следующий вопрос: насколько cart и purchase действительно связаны с просмотром на уровне `user-item` пар. Для архитектуры системы важно понять, стоит ли трактовать cart как промежуточную ступень воронки или как отдельный шумный event.


In [ ]:
funnel = pd.DataFrame(notebook_report['funnel_user_item_pairs'])
rates = pd.DataFrame(notebook_report['funnel_rates_pct'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=funnel, x='stage', y='pairs', hue='stage', palette='crest', legend=False, ax=axes[0])
axes[0].set_title('Пары user-item по этапам воронки')
axes[0].set_xlabel('stage')
axes[0].set_ylabel('pairs')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=rates, x='transition', y='rate_pct', hue='transition', palette='flare', legend=False, ax=axes[1])
axes[1].set_title('Конверсии между стадиями, %')
axes[1].set_xlabel('transition')
axes[1].set_ylabel('rate, %')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()

display(funnel)
display(rates)


**Выводы блока**

- Просмотр почти обязателен как верхняя стадия воронки, но переход из `view` в `addtocart` происходит лишь для небольшой доли `user-item` пар.
- Переход из `addtocart` в `transaction` уже намного плотнее, поэтому cart действительно отражает намерение, а не случайный клик.
- Это поддерживает схему, где `addtocart` является основным target, а `transaction` усиливает веса и участвует в post-filtering.


## 5. Разреженность по пользователям и товарам

Большая часть архитектурных ограничений возникает не из абсолютного числа строк, а из формы распределения взаимодействий: насколько коротка история пользователя и как часто один товар встречается в логах.


In [ ]:
user_bucket = pd.DataFrame(notebook_report['user_activity_buckets'])
item_bucket = pd.DataFrame(notebook_report['item_activity_buckets'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
sns.barplot(data=user_bucket, x='activity_bucket', y='share', color='#4C78A8', ax=axes[0])
axes[0].set_title('Распределение пользователей по числу событий')
axes[0].set_xlabel('events per user')
axes[0].set_ylabel('share')

sns.barplot(data=item_bucket, x='activity_bucket', y='share', color='#F58518', ax=axes[1])
axes[1].set_title('Распределение товаров по числу событий')
axes[1].set_xlabel('events per item')
axes[1].set_ylabel('share')
plt.tight_layout()

sparsity_summary = pd.DataFrame([
    {'metric': 'median_events_per_user', 'value': eda_summary['user_stats']['median_events_per_user']},
    {'metric': 'p95_events_per_user', 'value': eda_summary['user_stats']['p95_events_per_user']},
    {'metric': 'p99_events_per_user', 'value': eda_summary['user_stats']['p99_events_per_user']},
    {'metric': 'median_events_per_item', 'value': eda_summary['item_stats']['median_events_per_item']},
    {'metric': 'p95_events_per_item', 'value': eda_summary['item_stats']['p95_events_per_item']},
    {'metric': 'p99_events_per_item', 'value': eda_summary['item_stats']['p99_events_per_item']},
])

display(user_bucket)
display(item_bucket)
sparsity_summary


**Выводы блока**

- Более 70% пользователей имеют ровно одно событие, поэтому сложные персонализированные модели неизбежно сталкиваются с экстремально короткой историей.
- По товарам распределение тоже длиннохвостое, но заметно менее экстремальное: часть товаров имеет устойчивый объём взаимодействий, достаточный для candidate generation.
- Dense user-item representation здесь не даёт естественного выигрыша: матрица будет почти полностью пустой при очень слабой истории большинства пользователей.


## 6. Long-tail и repeat-interest

После общей разреженности важно разделить две разные гипотезы: насколько сильно помогают популярные товары как fallback и насколько часто пользователи возвращаются к уже знакомым товарам.


In [ ]:
long_tail = pd.DataFrame(notebook_report['long_tail_curve'])
repeat_interest = pd.DataFrame(notebook_report['repeat_interest'])
repeat_interest['value_share'] = repeat_interest['value_pct'] / 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=long_tail, x='top_n', y='event_share_pct', hue='top_n', palette='mako', legend=False, ax=axes[0])
axes[0].set_title('Сколько событий объясняют самые частые товары')
axes[0].set_xlabel('top N items')
axes[0].set_ylabel('event share, %')

sns.barplot(data=repeat_interest, x='metric', y='value_pct', hue='metric', palette='rocket', legend=False, ax=axes[1])
axes[1].set_title('Сигналы повторного интереса, %')
axes[1].set_xlabel('metric')
axes[1].set_ylabel('value, %')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()

display(long_tail)
repeat_interest[['metric', 'value_pct']]


**Выводы блока**

- Top-100 товаров объясняют лишь небольшую долю событий, поэтому один глобальный popularity baseline заведомо недостаточен.
- Одновременно long-tail не хаотичен: популярные товары всё равно нужны как обязательный fallback и как элемент cold-start стратегии.
- Повторные просмотры существенно заметнее повторных cart-событий, что поддерживает идею history-based ranking по недавним сигналам интереса.
- Низкая доля повторных cart-ов говорит в пользу мягкой персонализации: повторять интерес можно, но нельзя считать cart-механику полностью инерционной.


## 7. Покрытие и ограничения `item_properties`

Свойства товаров выглядят привлекательным источником признаков, но полезны только при аккуратном использовании. Важно оценить не только coverage по каталогу, но и coverage именно по событиям, особенно по ценным событиям `addtocart` и `transaction`.


In [ ]:
properties = notebook_report['item_properties_coverage']
coverage_table = pd.DataFrame([
    {'metric': 'event_item_coverage_pct', 'value_pct': properties['event_item_coverage_pct']},
    {'metric': 'event_row_coverage_pct', 'value_pct': properties['event_row_coverage_pct']},
    {'metric': 'addtocart_row_coverage_pct', 'value_pct': properties['addtocart_row_coverage_pct']},
    {'metric': 'transaction_row_coverage_pct', 'value_pct': properties['transaction_row_coverage_pct']},
    {'metric': 'available_zero_share_pct', 'value_pct': properties['available_zero_share_pct']},
])
top_coverage = pd.DataFrame(properties['top_item_snapshot_coverage'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=coverage_table, x='metric', y='value_pct', hue='metric', palette='viridis', legend=False, ax=axes[0])
axes[0].set_title('Покрытие snapshot по событиям и availability')
axes[0].set_xlabel('metric')
axes[0].set_ylabel('value, %')
axes[0].tick_params(axis='x', rotation=25)

sns.lineplot(data=top_coverage, x='top_n', y='snapshot_item_coverage', marker='o', ax=axes[1])
axes[1].set_title('Покрытие snapshot среди наиболее частых товаров')
axes[1].set_xlabel('top N items by events')
axes[1].set_ylabel('snapshot item coverage')
plt.tight_layout()

display(coverage_table)
pd.DataFrame([
    {'metric': 'items_only_in_events', 'value': properties['items_only_in_events']},
    {'metric': 'items_only_in_properties', 'value': properties['items_only_in_properties']},
    {'metric': 'median_versions_per_item_property', 'value': eda_summary['item_properties']['median_versions_per_item_property']},
    {'metric': 'p95_versions_per_item_property', 'value': eda_summary['item_properties']['p95_versions_per_item_property']},
])


**Выводы блока**

- Полного покрытия каталога по `item_properties` нет, поэтому модель не может зависеть от item features как от обязательного источника.
- Для ценных событий coverage заметно выше, чем для каталога в целом, значит последние `categoryid` и `available` полезны как лёгкий фильтр и контекст.
- Историчность `item_properties` требует snapshot-логики: использование будущих версий свойств привело бы к утечке.
- Широкий feature engineering по всем 1100+ свойствам в проекте не оправдан: отдача сомнительна, а стоимость по памяти и времени резко растёт.


## 8. Memory-safe инженерные ограничения

Ноутбук сознательно работает не с raw full join, а с узким набором parquet-артефактов, совпадающих с production pipeline. Это не косметическое упрощение, а часть архитектурного решения.


In [ ]:
memory_footprint = pd.DataFrame(notebook_report['memory_footprint'])
dtype_table = (
    pd.DataFrame(notebook_report['dtype_notes'])
    .fillna('')
    .rename_axis('column')
    .reset_index()
)

profile_quantiles = pd.DataFrame([
    {'metric': metric, 'value': value}
    for metric, value in notebook_report['profile_size_quantiles'].items()
])

display(memory_footprint)
display(profile_quantiles)
dtype_table


**Выводы блока**

- Рабочие артефакты остаются компактными за счёт downcasting типов и агрегации до уровня `user-item` перед построением item-to-item связей.
- `events_prepared`, `aggregated_train` и `item_snapshot` вместе помещаются в комфортный локальный footprint, что делает ноутбук воспроизводимым без memory-heavy приёмов.
- Отказ от full join `events x item_properties` и dense matrix здесь является не компромиссом качества, а рациональной инженерной границей.


## 9. Выводы для проектирования рекомендательной системы

1. Основной таргет offline-оптимизации: будущие `addtocart`, дополненные `transaction` как более сильным позитивным сигналом.
2. Split должен быть строго `time-based`, иначе temporal leakage завысит качество.
3. Базовый стек моделей должен включать `global_popularity`, `history-based baseline` и лёгкий `item-to-item retrieval`.
4. Сервису необходим fallback, потому что у большинства пользователей слишком короткая история для сложной персонализации.
5. `item_properties` стоит использовать точечно: snapshot по `categoryid` и `available`, без широкого и тяжёлого feature flattening.
6. Финальная production-архитектура должна опираться на компактные parquet artifacts, а не на дорогие online joins или dense offline representations.
